In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Optional, Dict, Callable, Tuple

import torch
import torch.nn as nn


In [1]:
import torch

x = torch.tensor([-2.0, -0.5, 0.2, 3.0, 6.0])

knots = torch.tensor([-6.0, 0.0, 6.0])          # 两段：[-6,0) 和 [0,6]
slopes = torch.tensor([0.0, 1.0])               # 第一段 y=0*x+0，第二段 y=1*x+0
intercepts = torch.tensor([0.0, 0.0])

x_clamped = torch.clamp(x, knots[0].item(), knots[-1].item())
y = torch.empty_like(x_clamped)

for i in range(len(slopes)):
    left, right = knots[i], knots[i+1]
    if i < len(slopes) - 1:
        mask = (x_clamped >= left) & (x_clamped < right)
    else:
        mask = (x_clamped >= left) & (x_clamped <= right)

    y_piece = slopes[i] * x_clamped + intercepts[i]
    print(f"segment {i}: [{left.item()}, {right.item()}], mask={mask.tolist()}, y_piece={y_piece.tolist()}")

    y = torch.where(mask, y_piece, y)

print("final y:", y.tolist())


segment 0: [-6.0, 0.0], mask=[True, True, False, False, False], y_piece=[0.0, 0.0, 0.0, 0.0, 0.0]
segment 1: [0.0, 6.0], mask=[False, False, True, True, True], y_piece=[-2.0, -0.5, 0.20000000298023224, 3.0, 6.0]
final y: [0.0, 0.0, 0.20000000298023224, 3.0, 6.0]


In [ ]:
def poly_eval(x: torch.Tensor, coeffs: torch.Tensor) -> torch.Tensor:
    """Horner: coeffs = [c0, c1, ..., ck] for sum_{i=0}^k c_i x^i"""
    #创建一个与x同形状的全0张量
    y = torch.zeros_like(x)
    for c in reversed(coeffs):
        y = y * x + c
    return y

In [ ]:
def pwl_eval_sim(x: torch.Tensor, knots: torch.Tensor, slopes: torch.Tensor, intercepts: torch.Tensor) -> torch.Tensor:
    """
    Piecewise linear (SIM ONLY):
    knots: [K+1] increasing, define K segments [k0,k1),...,[kK-1,kK]
    slopes/intercepts: [K]
    """
    # clamp to range
    x_clamped = torch.clamp(x, knots[0].item(), knots[-1].item())

    y = torch.empty_like(x_clamped)
    # naive loop; later you can vectorize if needed
    for i in range(len(slopes)):
        left, right = knots[i], knots[i + 1]
        mask = (x_clamped >= left) & (x_clamped < right) if i < len(slopes) - 1 else (x_clamped >= left) & (x_clamped <= right)
        y = torch.where(mask, slopes[i] * x_clamped + intercepts[i], y)
    return y

In [ ]:
class HEGELU(nn.Module):
    """
    HE-friendly GELU replacement.
    approx:
      - "poly": polynomial approximation (HE-friendly)
      - "pwl": piecewise linear (SIM ONLY)
      - "identity": y=x (debug)
    """
    def __init__(
        self,
        approx: str = "poly",
        poly_coeffs: Optional[torch.Tensor] = None,
        pwl_knots: Optional[torch.Tensor] = None,
        pwl_slopes: Optional[torch.Tensor] = None,
        pwl_intercepts: Optional[torch.Tensor] = None,
    ):
        super().__init__()
        self.approx = approx

        if poly_coeffs is None:
            # placeholder; replace with your fitted coeffs
            poly_coeffs = torch.tensor([0.0, 0.5, 0.0, 0.0], dtype=torch.float32)
        self.register_buffer("poly_coeffs", poly_coeffs)

        # SIM ONLY
        if pwl_knots is None:
            pwl_knots = torch.tensor([-6.0, 0.0, 6.0], dtype=torch.float32)
        if pwl_slopes is None:
            pwl_slopes = torch.tensor([0.0, 1.0], dtype=torch.float32)
        if pwl_intercepts is None:
            pwl_intercepts = torch.tensor([0.0, 0.0], dtype=torch.float32)

        self.register_buffer("pwl_knots", pwl_knots)
        self.register_buffer("pwl_slopes", pwl_slopes)
        self.register_buffer("pwl_intercepts", pwl_intercepts)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.approx == "identity":
            return x
        if self.approx == "poly":
            return poly_eval(x, self.poly_coeffs)
        if self.approx == "pwl":
            return pwl_eval_sim(x, self.pwl_knots, self.pwl_slopes, self.pwl_intercepts)
        raise ValueError(f"Unknown approx: {self.approx}")

# ----------------------------
# LayerNorm
# ----------------------------

In [ ]:
class HELayerNorm(nn.Module):
    """
    HE-friendly LayerNorm replacement.

    approx:
      - "affine_only": y = x * gamma + beta
      - "static_calib": y = (x - mu) * inv_std * gamma + beta
    """
    def __init__(
        self,
        normalized_shape,
        eps: float = 1e-5,
        elementwise_affine: bool = True,
        approx: str = "affine_only",
        mu: Optional[torch.Tensor] = None,
        inv_std: Optional[torch.Tensor] = None,
    ):
        super().__init__()
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = tuple(normalized_shape)
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        self.approx = approx

        if elementwise_affine:
            self.weight = nn.Parameter(torch.ones(self.normalized_shape))
            self.bias = nn.Parameter(torch.zeros(self.normalized_shape))
        else:
            self.register_parameter("weight", None)
            self.register_parameter("bias", None)

        # static calibration buffers (optional)
        if mu is None:
            mu = torch.zeros(self.normalized_shape, dtype=torch.float32)
        if inv_std is None:
            inv_std = torch.ones(self.normalized_shape, dtype=torch.float32)
        self.register_buffer("mu", mu)
        self.register_buffer("inv_std", inv_std)

    @torch.no_grad()
    def set_static_calibration(self, mu: torch.Tensor, inv_std: torch.Tensor):
        self.mu.copy_(mu)
        self.inv_std.copy_(inv_std)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.approx == "affine_only":
            if self.elementwise_affine:
                return x * self.weight + self.bias
            return x

        if self.approx == "static_calib":
            # NOTE: no runtime mean/var; only constant shift/scale
            y = (x - self.mu) * self.inv_std
            if self.elementwise_affine:
                y = y * self.weight + self.bias
            return y

        raise ValueError(f"Unknown approx: {self.approx}")